# Атака RSA Blinding
## Введение
RSA (названа в честь своих создателей Ronald Linn Rivest, Adi Shamir and Leonard Adleman) - это пример асимметричной криптосистемы, которая может быть использована для безопасной передачи данных и создания подписей. Хоть от неё постепенно избавляются, её всё ещё можно встретить во многих продуктах и системах, поэтому есть смысл понимать, как она работает. Начнем с основ

## Группы и Поля
Группа - это множество $\mathbb{G}$ определенной на нем операцией с двумя аргументами $*, \forall a,b \in \mathbb{G}\ \Rightarrow a*b \in \mathbb{G}$ и свойствами:

+ Ассоциативности ($\forall a,b,c\in \mathbb{G},\ (a*b)*c = a*(b*c)$)

+ Наличия единичного элемента ($\exists\ e \in \mathbb{G}:\forall a \in \mathbb{G}, \ a*e = e*a = a$)

+ Наличия обратного элемента ($\forall a\in \mathbb{G},\ \exists b \in \mathbb{G}: a*b=e=b*a$)

Группа называется коммутативной (абелевой), если она удовлетворяет свойству коммутативности:

+ $\forall a,b \in \mathbb{G},\ a*b=b*a$

Т.е., если у нас есть некоторое множество, результат применения оператора к любым двум элементам также содержится в этом множестве, порядок вычисления выражений не важен, есть некий элемент, который не изменяет другие элементы (1) и для любого элемента всегда есть такой парный, что при применении к ним оператора получается единичный (1), то у нас есть группа. Если ещё и можно поменять местами аргументы при вызове оператора и ничего не изменится, то аж абелева.

Если мы используем аддитивную нотацию (ставим плюс), то группа называется аддитивной. Если мультипликативную (ставим знак умножения) - мультипликативной.

Давайте рассмотрим простой пример аддитивной группы целых чисел по модулю некоторого числа $n$. Например, если $n=5$, то группа содержит элементы $\{0,1,2,3,4\}$. Если происходит переполнение (результат меньше $0$ или больше или равен $n$), мы тут же добавляем или вычитаем $k*n, k \in \mathbb{N}$ из результата, чтобы он снова был в множестве. Как видно, $0$ - единичный элемент, $1$ - обратный элемент к $4$, а $2$ - к $3$. Ассоциативность очевидна, а поскольку мы легко можем менять элементы местами при сложении, то эта группа ещё и абелева.

О полях можно думать как о группах с двумя операциями (это не совсем верно, но так проще вникнуть). Допустим у Вас есть множество с двумя операциями $(+,*)$. Оно будет полем, если:

1. Для обоих операций оно является абелевой группой, за исключением единичного элемента аддитивной группы, который не входит в мультипликативную (у $0$ же не может быть обратного элемента).

2. Действует закон дистрибутивности: $a*(b+c)= a*b+a*c$.

В качестве примера давайте посмотрим на поле $F_n$, где $n=5$:

1. Сложение осталось таким же, как и в прошлом примере

2. Таблица умножения: 


|     | 0   | 1   | 2   | 3   | 4   |
| --- | --- | --- | --- | --- | --- |
| 0   | 0   | 0   | 0   | 0   | 0   |
| 1   | 0   | 1   | 2   | 3   | 4   |
| 2   | 0   | 2   | 4   | 1   | 3   |
| 3   | 0   | 3   | 1   | 4   | 2   |
| 4   | 0   | 4   | 3   | 2   | 1   |

Можно видеть, что в каждом ненулевом ряду есть по $1$, так что у каждого элемента есть обратный. Если удалить ряд и столбец, содержащие нулевой элемент, то как раз получится таблица мультипликативной группы по модулю $n$ или $\mathbb{Z}_n^{*}$, которая содержит элементы $\{1,2,3,4\}$. Здесь же можно заметить интересную особенность групп. Если порядок (количество элементов) в группе не простое, то можно генерировать подгруппы. Это группы, которые используют ту же операцию, что и оригинальная (например, умножение по модулю $5$), но состоят из подмножества элементов оригинальной группы. В данном случае элементы $\{1,4\}$ образуют такую подгруппу, т.к.  $4\cdot 4 = 1\ mod\ 5$ (получаем замкнутое по умножению множество). Подгруппы упрощают вычисление дискретного логарифма (но об это позже). Из-за этого в криптографии часто используются большие безопасные простые числа $p=2*q+1$, где $q$ - это другое простое число. Таким образом получается всего 2 нетривиальных подгруппы с порядками $2$ и $q$.

## RSA
RSA использует мультипликативную группу по модулю $N=pq$, где $p$ и $q$ - простые числа. Степень мультипликативной группы (по-сути, её мощность) может быть вычислена при помощи функции Эйлера  для составного числа из двух простых: $\varphi(N)=(p-1)(q-1)$. Функция считает количество натуральных чисел меньше $N$, которые не кратны $p$ или $q$. 

Например, если мы возьмем $N=13\cdot 17=221$, то элемент $26$ не состоит в мультипликативной группе, т.к. $26\cdot 17 = 0\ \mathit{mod}\ 221$, а $0$ не в группе. Т.е. надо исключить все элементы кратные $p$ (таких $q$) и $q$ (таких $p$). Т.е. всего элементов в группе будет $p\cdot q - p - q +1=(p-1)\cdot(q-1)$ (добавляем $1$, потому что $0$ посчитали до этого два раза).

Поскольку все остальные числа взаимно просты с $N$, они состоят в мультипликативной группе. Как мы знаем, если возвести любой элемент конечной мультипликативной группы в степень этой группы, то получим нейтральный элемент (единицу): $a^{\varphi(N)}=1, a \in Z^{*}_{N}$. Поэтому в RSA используют два числа $e$ (открытая экспонента) и $d$ (закрытая экспонента), такие что $ed=1\ mod\ \varphi(N)$. Пара чисел $(e,N)$ используется как открытый ключ, а $(d,N)$ как закрытый. Вычисление $d$ из открытого ключа является сверхполиномиальной задачей (NP), если не были сгенерированы слабые $N$, $d$ или $e$. Одним из способов решения является факторизация $N$ в произведение $p$ и $q$.

Пусть дан открытый текст (число) $M, M < N$, открытый ключ $(e,N)$ и закрытый ключ $(d,N)$,  шифрование и расшифрование осуществляются следующим образом:

Шифрование
$C=M^{e}\ mod\ N$

Расшифрование

$M=C^{d} \ mod\ N$

Проверка корректности:

$C^{d}\ mod\ N=M^{ed}\ mod\ N= M^{ed\ \mathit{mod}\ \varphi(N)}\ mod\ N=M^{1}\ mod\ N= M\ mod\ N$

## Подготовка
Попробуем немного поработать с RSA. Если ещё не установили, установите Pycryptodome. На Linux и Windows должна сработать следующая команда (Предварительно надо установить python 3 и pip, но я надеюсь, что вы справились с этим самостоятельно):

In [1]:
!python3 -m pip install --user pycryptodome

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 11.3 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


После установки надо перезапустить ядро jupyter (круговая стрелка рядом с "Run"). Если возникнут проблемы, загляните в документацию: [Pycryptodome installation](https://pycryptodome.readthedocs.io/en/latest/src/installation.html).

## Примитивная реализация RSA
Давайте сделаем простейшую версию RSA. Будем использовать открытую экспоненту $e=65537$. Обычно используют эту константу, потому что она переполняет модуль даже при открытом тексте $M=2$ и у нее удобное двоичное представление $65537_{10}=10000000000000001_{2}$, которое позволяет эффективно возводить число в степень, используя алгоритм "Square and multiply".

Сначала сгенерируем $p$ и $q$. Функция getStrongPrime дает возможность выбрать количество бит в генерируемом простом числе и проверяет, что $НОД(p-1,e)=1$

In [1]:
try:
    from Crypto.Util.number import getStrongPrime, inverse,bytes_to_long, long_to_bytes
except ImportError:
    print ("Pycryptodome not installed")

In [2]:
e=65537
p=getStrongPrime(1024,e=e)
q=getStrongPrime(1024,e=e)

assert p.bit_length() == 1024
assert q.bit_length() == 1024

print(f"p: {p}")

p: 179023994394881506077037689968564098724818791525506249327637257170378990965253180885667561295863240769706366143697622090015281698350003350990474327042316367805050198786668452400555190990411916482445461808851047670236385105278119489888154297974480560785284161763485043264565870450072179637254740493220251090133


In [3]:
N=p*q
phi=(p-1)*(q-1)
d=inverse(e,phi)
public_key=(e,N)
private_key=(d,N)

print(f"pk: {public_key}")
print(f"sk: {private_key}")

pk: (65537, 30812096889355278821405656297256171012089417804594818766411519996333814716566743466862369752339442162849537114898160360324824641432439691357547513040753556093107266500200621193149503114948769584772916470031666461491913889740352225119809373143509495722204662003297005668805902517899796866891849577163312856532351497049512780365920065500894575901075496959757555820991744614053139076399614401218852696152374638409046997568787639338056227064843469587037675652393350192129609399559462972565422810334362355350767035881485251524984932402946376161186865633306428368048158572069979102196350347309483460417092100558787634676477)
sk: (2940165255000398357651777664314065591319462168310948290576433504754150740695995294546854474879917748047853125035302583965749971163252027066278512631078024836954161962132911557678853893677695353041707550370585051958923690316450870656754991483094516584493511982086737024139211019366006372987496524111665312890339360372829045430476969332946592403747045695529451873771

Мы успешно сгенерировали ключи, теперь давайте зашифруем сообщение, расшифруем закрытый текст и проверим, что получили то же самое

In [4]:
M=bytes_to_long(b'Hello, RSA!')
print(f"Message: {M}\nType: {type(M)}\n")

# Exponentiating (Encrypt)
C=pow(M,e,N)


print(f'CipherText (Hex): {hex(C)}\nType: {type(C)}\n')

# Exponentiating (Decrypt)
M1=pow(C,d,N)

assert M1==M
print ('M1:',long_to_bytes(M1))

Message: 87521618088895491219865889
Type: <class 'int'>

CipherText (Hex): 0x52267bb1530dd10291f7d3e88fe84dc7b12d868511656aa32a65b4a3d1727237d18359b6e3d6bed091740c4d43b0600991df11e7af725519ebb62f31e3ee66ac005858b3af2607680c0ea263e34a387887f561cff1e58a0c1121f98a3ced643f987327a215ccf5e797677114d55e1adacba3c1d079afebc65f88cd340d7fa05f383d9c719d6425891d10c09958225be5b4bf90ccab77ac0ac2cd2474c5cf19f4664ef3d4fbd89dc57db5352bf9eb3f7d98dc7eb8f0e67aff27aa13a428e6578b73a2762b6dce95bae5f7248705909faa1cf0572eff4cae67416dda4d60b1522550d9dbe2c5d0311f359017f238f37d636e404e953c325ae52abc284d95ad00bb
Type: <class 'int'>

M1: b'Hello, RSA!'


Создание подписи - обратная операция к шифрованию. 
$$Sign(M)\equiv Dec(M),\space Check(S) \equiv Enc(S)$$
Таким образом любой, владеющий открытым ключом, может проверить правильность подписи, а создать её может только сторона, у которой есть закрытый ключ.
Поздравляю, теперь вы знаете, как шифровать и создавать подписи при помощи RSA. Дальше рассмотрим одно из его интересных свойств.

## RSA Blinding
RSA - это гомоморфное шифрование по отношению к операции умножения.
Отображение является гомоморфизмом групп, если оно сохраняет отношения между элементами. Если ничего не понятно, не беспокойтесь, я в первый раз, когда услышал, тоже ничего не понял. Что это значит на практике: пусть у вас есть два элемента группы $G_1$ $(x,y)$ и вы применяете к ним гомоморфное отображение, они будут также связаны в новой группе $G_2$ (для RSA $G_1= G_2$): 
$$\varphi(x\cdot y)=\varphi(x)\times\varphi(y)$$
Для шифрования RSA: $$Enc(M_1 \cdot M_2)=Enc(M_1)\times Enc(M_2)$$
То же самое верно и для расшифрования:
$$Dec(C_1 \times C_2)=Dec(C_1) \cdot Dec(C_2)$$
Протестируем это свойство в python

In [5]:
class BasicRSA:
    def __init__(self, e,p,q):
        self.e=e
        self.p=p
        self.q=q
        self.N=p*q
        self.d=inverse(e, (p-1)*(q-1))
    
    def encryptNumber(self, m):
        return pow(m, self.e, self.N)
    
    def decryptNumber(self, c):
        return pow(c, self.d, self.N)

base_rsa = BasicRSA(e,p,q) #we created these parameters earlier

m1 = 2 # Message 1
m2 = 3 # Message 2
m3 = m1*m2 # Message 3

c1 = base_rsa.encryptNumber(m1) # CipherText1
c2 = base_rsa.encryptNumber(m2) # CipherText2

print('c1:',c1)
print('c2:',c2)

c3 = (c1*c2) % base_rsa.N # CipherText 3
print('c3:',c3)

m3_dec = base_rsa.decryptNumber(c3)
print ('m3: %d, m3_dec: %d'%(m3,m3_dec))

assert m3_dec==m3

c1: 13076596010590422575549012034011493612252249576613446514063268734551415461386162928399446591875576430086873575169921150532779297244296710152988530111061559697569276292606445779790313211476898029350429957689240703051158017618857868034799767738438386163296492949499365056041560353690209774044521919046813367974127223753264236906159045355840593336756144888267779918465270062672039618539946699302384881002922043733293067278989748928035414217013409167917427131848796109294276491388955113321607769542378609541271770906272717256770177905445231935302209473487407601210120546293417620018835248796355450827382013623011390206055
c2: 19174136994943218092975594323266527657646150444687502135283503384497812449408616677350574133950619558549047716838348875715474448805785811168478051103810103772614587580809493471082028089338644416667813014393204068663976413039341610594242319527030708702135411682484548980607400192347662310273866172074109894883778574955861750665565469895003302922132596816028919863359880097485

## Атакуем сервер
Теперь попробуйте применить эти знания к уязвимому серверу. Вы можете приконнеrтиться, используя ```nc cryptotraining.zone 1337``` или при помощи питоновских сокетов.

In [6]:
import socket
import re
class VulnServerClient:
    def __init__(self,show=True):
        """Ининциализация, подключаемся к серверу"""
        self.s=socket.socket(socket.AF_INET,socket.SOCK_STREAM)
        self.s.connect(('cryptotraining.zone',1337))
        if show:
            print (self.recv_until().decode())
    def recv_until(self,symb=b'\n>'):
        """Получение сообщения с сервера, по умолчанию до приглашения к вводу команды"""
        data=b''
        while True:
            
            data+=self.s.recv(1)
            if data[-len(symb):]==symb:
                break
        return data
    def get_public_key(self,show=True):
        """Получение открытого ключа с сервера"""
        self.s.sendall('public\n'.encode())
        response=self.recv_until().decode()
        if show:
            print (response)
        e=int(re.search(r'(?<=e: )\d+',response).group(0))
        N=int(re.search(r'(?<=N: )\d+',response).group(0))
        self.num_len=len(long_to_bytes(N))
        return (e,N)
    
    def signBytes(self,m,show=True):
        """Получение подписи для выбранного сообщения в байтах с сервера"""
        try:
            num_len=self.num_len
        except AttributeError:
            print ('You need to get the public key from the server first')
            return
        if len(m)>num_len:
            print ("The message is too long")
            return
        if len(m)<num_len:
            m=bytes((num_len-len(m))*[0x0]) +m
        hex_m=m.hex().encode()
        self.s.sendall(b'sign '+hex_m+b'\n')
        response=self.recv_until().decode()
        if show:
            print (response)
        if response.find('flag')!=-1:
            print('You tried to submit \'flag\'')
            return None
        signature_hex=re.search(r'(?<=Signature: )[0-9a-f]+',response).group(0)
        signature_bytes=bytes.fromhex(signature_hex)
        return bytes_to_long(signature_bytes)
    
    
    def signNumber(self,m,show=True):
        """Получение подписи с сервера для выбранного сообщения в числовом представлении"""
        try:
            num_len=self.num_len
        except AttributeError:
            print ('You need to get the public key from the server first')
            return
        return self.signBytes(long_to_bytes(m,num_len),show)
        
    def checkSignatureNumber(self,c,show=True):
        """Проверка сигнатуры (на сервере) для подписи в числовом представлении"""
        try:
            num_len=self.num_len
        except AttributeError:
            print ('You need to get the public key from the server first')
            return
        signature_bytes=long_to_bytes(c,num_len)
        self.checkSignatureBytes(signature_bytes,show)
    
    def checkSignatureBytes(self,c,show=True):
        """Проверка сигнатуры (на сервере) для подписи в байтовом представлении"""
        try:
            num_len=self.num_len
        except AttributeError:
            print ('You need to get the public key from the server first')
            return
        if len(c)>num_len:
            print ("The message is too long")
            return
        
        hex_c=c.hex().encode()
        self.s.sendall(b'flag '+hex_c+b'\n',)
        response=self.recv_until(b'\n').decode()
        
        if show:
            print (response)
        
        if response.find('Wrong')!=-1:
            print('Wrong signature')
            x=self.recv_until()
            if show:
                print (x)
            return
        flag=re.search(r'CRYPTOTRAINING\{.*\}',response).group(0)
        print ('FLAG: ',flag)
        
    def __del__(self):
        self.s.close()

In [7]:
vs=VulnServerClient()
(e,N)=vs.get_public_key()

Welcome to RSA blinding task
Available commands:
help - print this help
public - show public key
sign <hex(data)> - sign data
flag <hex(signature(b'flag'))> - print flag 
quit - quit
>
e: 65537
N: 20159717663186764200842482638329142432479376755681286432561400011207751568770239378735042390550988864636478212097889382541806378632813451522011734778394352464750695430236459156439656932108536936107092785759187120915559173321302027525229018106368725032056109022369913503577023942696069608771010384365856481001383579432844112231215767630328627015097422540087789462404508697086321213990868031273219614897901436844999442259387453021270642395531884848697650933478124254071912232445708062597679170291021925633789812405697682134528381868778865376836541179591638312152472136313757252384761293684336082840137773984575947459061
>


Вы можете подписывать сообщения при помощи методов signNumber (подписать число) и signBytes (подписать сообщение из байтов)

Проверять подпись можете при помощи методов checkSignatureNumber и checkSignatureBytes.

Ваша цель - получить правильную подпись для сообщения 'flag'.

Помните, что RSA - это гомоморфизм и решите задание.

Удачи!

In [8]:
vs.signBytes(b'flag')

I found 'flag' in your message for signing. Despicable...
>
You tried to submit 'flag'


Используем свойство гомоморфизма RSA:

$RSA_{Sign}(msg \cdot r \mod N) = RSA_{Sign}(msg) \cdot RSA_{Sign}(r) \mod N$

Значит, если сервер блокирует ```flag``` по байтам, то нужно добавить ```r,``` то есть случайность, которую мы знаем.

Сервер подпишет сообщение и зная $r^{-1}$, мы сможем получить корректную подпись и для ```msg.```

In [9]:
import random

msg = b'flag'
m = bytes_to_long(msg)

r = random.randrange(2, N)
r_inv = inverse(r, N)

blinded = (m * pow(r, e, N)) % N
blinded_bytes = long_to_bytes(blinded, vs.num_len)

# Signature
s_prod = vs.signBytes(blinded_bytes)

Signature: 00d4ead15aee380aa1320d49aadd1f4a35ff097482aab2f1357cffe726df41837072ca0c4fe31b79f88d1390c99cb672aa6c28f99aa96fbff37b954eb808aef663d0e142da5a803db5b11ec0bc5b2a5988a4b8705cef71295140341541c87c4fe16fe4ce5a02c27cf143c5d7b5f5684b149fc5bbe179a32403147ecf8c5c793c50a4f60284d29b66bf93898411efde35e66894e443b4de38c849d4bb83655abddb1ae2d4b860b1d97885eeb1a3c74e63e72d7d1b4e5547ac64f2c3ef8aa69bad4b3dc5719583df8e4c60f3c1fe8624435b5bcc459577711538dd3376951ce24c6c6c2861d69432e455a3452275b4dfdbaf3548cc3b0e776196e48bab7ddcf6d0
>


Подпись произведения получена. Получаем для сообщения:

In [10]:
signature = (s_prod * r_inv) % N

Проверка

In [11]:
vs.checkSignatureNumber(signature, show=True)

Congratulations! Here is your flag: CRYPTOTRAINING{n0t_s0_bl1nd_4ft3r_4ll}

FLAG:  CRYPTOTRAINING{n0t_s0_bl1nd_4ft3r_4ll}
